## Mess around video stuff

In [1]:
import sys
from PySide6.QtWidgets import QApplication

import qiskit_metal as qm

In [15]:
design = qm.designs.DesignPlanar()

design.overwrite_enabled = True
design.chips.main.size.size_x = '6.25mm'
design.chips.main.size.size_y = '6.25mm'
design.chips.main.size.size_z = '-270um'
design.chips.main.size.sample_holder_top = '50um'
design.chips.main.size.sample_holder_bottom = '800um'

gui = qm.MetalGUI(design)

gui.rebuild()
gui.autoscale()

In [3]:
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven

port_1  = LaunchpadWirebondDriven(design, 'port_1', options = dict(pos_x = '-2.496mm',
                                                                   pos_y = '-0mm',
                                                                   orientation = '0',
                                                                   pad_width = '350um',
                                                                   pad_height = '129 um',
                                                                   pad_gap = '150um',
                                                                   trace_width = '10um',
                                                                   trace_gap = '6um',
                                                                   lead_length = '9um',
                                                                   taper_height = '350um'))


port_2  = LaunchpadWirebondDriven(design, 'port_2', options = dict(pos_x = '2.496mm',
                                                                   pos_y = '-0mm',
                                                                   orientation = '180',
                                                                   pad_width = '350um',
                                                                   pad_height = '129 um',
                                                                   pad_gap = '150um',
                                                                   trace_width = '10um',
                                                                   trace_gap = '6um',
                                                                   lead_length = '9um',
                                                                   taper_height = '350um'))


gui.rebuild()
gui.autoscale()

In [4]:
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

bus_1 = RouteStraight(design,
                      'bus_1',
                      options= dict(hfss_wire_bonds = True,
                                    pin_inputs = dict(start_pin = dict(component = 'port_1', pin = 'tie'),
                                                      end_pin = dict(component = 'port_2', pin = 'tie')),
                                    trace_width = '10um',
                                    trace_gap = '6um'))

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

coupl_1 = CoupledLineTee(design, 'coupl_1', options= dict(pos_x = '-0.55mm',
                                                          pos_y = '0',
                                                          orientation = '180',
                                                          coupling_length = '650um',
                                                          down_length = '100um',
                                                          fillet = '50um',
                                                          coupling_space = '4um',
                                                          open_termination = True))

gui.rebuild()
gui.autoscale()

In [16]:
from qiskit_metal.analyses.em.cpw_calculations import guided_wavelength
def find_resonator_length(frequency, line_width, line_gap, N):
    [lambdaG, etfSart, q] = guided_wavelength(frequency, line_width,
                                              line_gap, substrate_thickness=350*10**-6,film_thickness = 200*10**-9)
    return str(lambdaG/N*10**3) + "mm"

def find_resonator_length_2(frequency, line_width, line_gap, N):
    [lambdaG, etfSart, q] = guided_wavelength(frequency, line_width,
                                              line_gap, substrate_thickness=350*10**-6,film_thickness = 200*10**-9)
    return lambdaG/N*10**3

round(find_resonator_length_2(5.0*10**9, 6*10**(-6), 10*10** (-6),2),2)

12.14

In [10]:
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

open_end = OpenToGround(design, 'open_end', options=dict(pos_x='-0.55mm',
                                                         pos_y ='2.25mm',
                                                        orientation='90', 
                                                        termination_gap= '6um',
                                                        width = '10um'))
gui.rebuild()
gui.autoscale()

In [11]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

res_1 = RouteMeander (design, 'res_1', options = dict(
                                                    pin_inputs = dict(start_pin = dict(component = 'coupl_1',
                                                                                pin = 'second_end'),
                                                                      end_pin=dict(component = 'open_end',
                                                                                       pin = 'open')),
                                                    lead = dict(start_straigt= '0um',
                                                               end_straight= '0um'),
                                                    meander=dict (spacing= '150um',
                                                                    asymmetry= '-300um'),
                                                    hfss_wire_bonds=True,
                                                    total_length= '12mm', 
                                                    trace_width= '10um', 
                                                    fillet= '50um', 
                                                    trace_gap='6um' ))
gui.rebuild()
gui.autoscale()

10:29AM 25s WARNING [__init__]: Component coupl_1 does not exist. res_1 has not been built. Please check your pin_input values.


In [12]:
coupl_2 = CoupledLineTee(design, 'coupl_2', options= dict(pos_x = '1mm',
                                                          pos_y = '0',
                                                          orientation = '0',
                                                          coupling_length = '650um',
                                                          down_length = '100um',
                                                          fillet = '50um',
                                                          coupling_space = '4um',
                                                          open_termination = True))

open_end_2 = OpenToGround(design, 'open_end_2', options=dict(pos_x='1mm',
                                                         pos_y ='-2.25mm',
                                                        orientation='90', 
                                                        termination_gap= '6um',
                                                        width = '10um'))

res_2 = RouteMeander (design, 'res_2', options = dict(
                                                    pin_inputs = dict(start_pin = dict(component = 'coupl_2',
                                                                                pin = 'second_end'),
                                                                      end_pin=dict(component = 'open_end_2',
                                                                                       pin = 'open')),
                                                    lead = dict(start_straigt= '0um',
                                                               end_straight= '0um'),
                                                    meander=dict (spacing= '150um',
                                                                    asymmetry= '-300um'),
                                                    hfss_wire_bonds=True,
                                                    total_length= '12mm', 
                                                    trace_width= '10um', 
                                                    fillet= '50um', 
                                                    trace_gap='6um' ))
gui.rebuild()
gui.autoscale()

10:29AM 26s WARNING [check_lengths]: For path table, component=res_2, key=trace has short segments that could cause issues with fillet. Values in (27-27)  are index(es) in shapely geometry.
10:29AM 26s WARNING [check_lengths]: For path table, component=res_2, key=cut has short segments that could cause issues with fillet. Values in (27-27)  are index(es) in shapely geometry.


In [13]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

q1 = TransmonPocket(design,'Q1', options=dict(pos_x='-2.5 mm',
                                                pos_y='2.25 mm',
                                                pad_width='425 um',
                                                pocket_height='650 um',
                                                connection_pads=dict(readout=dict(loc_W=-1, loc_H=-1, pad_width='100um'))
                                            ))
gui.rebuild()
gui.autoscale()

In [14]:
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond

coupl_qubit = CoupledLineTee(design, 'coupl_qubit', options= dict(pos_x = '-1.5mm',
                                                          pos_y = '0',
                                                          orientation = '180',
                                                          coupling_length = '650um',
                                                          down_length = '100um',
                                                          fillet = '50um',
                                                          coupling_space = '4um',
                                                          open_termination = True))

readout_res = RouteMeander(design, 'Readout_Res', options=dict(pin_inputs=dict(
                                                        start_pin=dict(component='Q1', pin='readout'),
                                                        end_pin=dict(component='coupl_qubit', pin='second_end')
                                                    ),
                                            lead=dict(start_straight='100um'),
                                            meander=dict(asymmetry='0um'),
                                            trace_width= '10um',
                                            trace_gap='6um',
                                            fillet='50um',
                                            total_length='12mm'
))
gui.rebuild()
gui.autoscale()

In [24]:
import math
from scipy.integrate import quad

def calc_f0(l, N):
    c = 3*10**8
    return c/()

def calc_k0(w,s):
    return w/(w+2*s)

def calc_k_prime_0(w,s):
    return (1-calc_k0(w,s))**0.5

def calc_k1(w,s,d):
    top = math.sinh((math.pi * w)/(4 * d))
    bottom = math.sinh((math.pi * (w+2*s))/(4 * d))
    return top/bottom

def calc_k_prime_1(w,s,d):
    return (1-calc_k1(w,s,d))**0.5

# def inner_K(k,x):
#     return 1/((1-k**2 * math.sin(x)**2)**0.5)

# def K(k):
#     result,_ = quad(inner_K(k),0,math.pi)
#     return result

def conv_epslion(epsilon):
    return (epsilon+1)/2

def a(in_freq, width, gap, N, epslion): 
    ep = conv_epslion(epslion)
    length = round(find_resonator_length_2(in_freq, width*10**(-6), gap*10**(-6), N),5)
    c = 2.99 * 10**8
    true_length = length * 10 ** (-3)
    f0 = c/(N * true_length * ep ** 0.5)
    return length, in_freq * 10 **(-9), f0 * 10 **(-9)


def calc_freq(in_freq, width, gap, N, epslion, in_len): 
    ep = conv_epslion(epslion)
    c = 2.99 * 10**8
    calc_length = round(find_resonator_length_2(in_freq, width*10**(-6), gap*10**(-6), N),5)
    true_len_in = in_len * 10 ** (-3)
    true_len_calc = calc_length * 10 ** (-3)
    f0_in = c/(N * true_len_in * ep ** 0.5)
    f0_calc = c/(N * true_len_calc * ep ** 0.5)
    return in_freq * 10 **(-9), f0_in * 10 **(-9), f0_calc * 10 **(-9)

#round(find_resonator_length_2(5.0*10**9, 6*10**(-6), 10*10** (-6),2),2)
calc_freq(5*10**(9),6,10,2,11.9,9)

(5.0, 6.540618612680254, 4.847671634178065)

## Making a quantum chip - via docs

In [2]:
design = qm.designs.DesignPlanar()

design.overwrite_enabled = True
design.chips.main.size.size_x = '6.25mm'
design.chips.main.size.size_y = '6.25mm'
design.chips.main.size.size_z = '-270um'
design.chips.main.size.sample_holder_top = '50um'
design.chips.main.size.sample_holder_bottom = '800um'

gui = qm.MetalGUI(design)

gui.rebuild()
gui.autoscale()

In [3]:
options = dict(
    pad_width="425 um",
    pocket_height="650um",
    gds_cell_name="FakeJunction_01",
    connection_pads=dict(
        a=dict(loc_W=+1, loc_H=+1),
        b=dict(loc_W=-1, loc_H=+1, ),#pad_height="30um"),
        c=dict(loc_W=+1, loc_H=-1, ),#pad_width="200um"),
        d=dict(loc_W=-1, loc_H=-1, ),#pad_height="50um"),
    ),
)

In [5]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

q_1 = TransmonPocket(
    design, "Q_1", 
    options=dict(pos_x="+2.55mm", pos_y="+0.0mm", **options)
)
q_2 = TransmonPocket(design, "Q_2",
    options=dict(pos_x="+0.0mm", pos_y="-0.9mm", orientation="90", **options),
)
q3 = TransmonPocket(design, "Q_3", 
    options=dict(pos_x="-2.55mm", pos_y="+0.0mm", **options)
)
q4 = TransmonPocket(design,"Q_4",
    options=dict(pos_x="+0.0mm", pos_y="+0.9mm", orientation="90", **options),
)

gui.rebuild()
gui.autoscale()

In [12]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

def connect(
    component_name: str,
    component1: str,
    pin1: str,
    component2: str,
    pin2: str,
    length: str,
    asymmetry: str,
    flip = False,):
    myoptions = RouteMeander(design, component_name, options=dict(pin_inputs=dict(
                                                        start_pin=dict(component=component1, pin=pin1),
                                                        end_pin=dict(component=component2, pin=pin2)
                                                    ),
                                            lead=dict(start_straight='130um'),
                                            meander=dict(asymmetry=asymmetry),
                                            fillet='90um',
                                            total_length=length,
                                            lead_direction_inverted = "true" if flip else "false"))
    return myoptions

cpw1 = connect("cpw1", "Q_1", "d", "Q_2", "c", "6.0 mm", "150um")
cpw2 = connect("cpw2", "Q_3", "c", "Q_2", "a", "6.1 mm", "-150um", flip=True)
cpw3 = connect("cpw3", "Q_3", "a", "Q_4", "b", "6.0 mm", "150um")
cpw4 = connect("cpw4", "Q_1", "b", "Q_4", "d", "6.1 mm", "-150um", flip=True)

cpw5 = connect("cpw5", "Q_1", "a", "Q_4", "c", "7 mm", "-150um", flip=True)


gui.rebuild()
gui.autoscale()

10:51AM 02s WARNING [check_lengths]: For path table, component=cpw5, key=trace has short segments that could cause issues with fillet. Values in (28-28)  are index(es) in shapely geometry.
10:51AM 02s WARNING [check_lengths]: For path table, component=cpw5, key=cut has short segments that could cause issues with fillet. Values in (28-28)  are index(es) in shapely geometry.
10:51AM 03s WARNING [check_lengths]: For path table, component=cpw5, key=trace has short segments that could cause issues with fillet. Values in (28-28)  are index(es) in shapely geometry.
10:51AM 03s WARNING [check_lengths]: For path table, component=cpw5, key=cut has short segments that could cause issues with fillet. Values in (28-28)  are index(es) in shapely geometry.


### Not working code ----------------------------------------------

In [37]:


#design.delete_component('res_1')
# def create_resonator(frequency, width, gap, lambda_number):
#     length = round(find_resonator_length_2(frequency * 10**6, width*10**(-6), gap*10**(-6), lambda_number))
#     res = RouteMeander (design, 'res', options = dict(
#                                                     pin_inputs = dict(start_pin = dict(component = 'coupl_1',
#                                                                                 pin = 'second_end'),
#                                                                       end_pin=dict(component = 'open_end',
#                                                                                        pin = 'open')),
#                                                     lead = dict(start_straigt= '0um',
#                                                                end_straight= '0um'),
#                                                     meander=dict (spacing= '100um',
#                                                                     asymmetry= '-300um'),
#                                                     hfss_wire_bonds=True,
#                                                     total_length= length, 
#                                                     trace_width= width, 
#                                                     fillet= '10um', 
#                                                     trace_gap= gap ))
# #the funciton takes in numbers of MHz, micrometers, microemter
# create_resonator(5 * 10**3, 10, 6, 2)
# gui.rebuild()
#gui.autoscale()

In [22]:
# hfss_renderer = design.renderers.hfss
# hfss_renderer.start()